In [1]:
import psycopg2
import datetime as dt
import numpy as np
import pandas as pd
import logging
import json

In [2]:
# Configure logging
logging.basicConfig(filename='create_table.log', level=logging.DEBUG, 
                    format='%(asctime)s %(levelname)s:%(message)s')

In [3]:
# Load database configuration from JSON
with open('../../db_config.json') as config_file:
    db_params = json.load(config_file)
 
# Connect to the database
try:
    conn = psycopg2.connect(**db_params)
    cursor = conn.cursor()
    logging.info('Connection to database established')
except OperationalError as e:
    logging.error('Connection to database could not be established.')

INFO: Connection to database established


## Create First Table

This is the table that will be used for timeseries like the LSTM and the ARIMA models.
It has a constant stream of time with intervals of 10 minutes. If multiple events happen within those 10 minutes, they all show up at the same time interval. If nothing happened, the columns with the driving data are left with empty values.

In [4]:
create_table_1 = """
CREATE TABLE IF NOT EXISTS group14_warehouse.timestream_table AS 
SELECT 
    safe.event_start,
    safe.event_end,
    RTRIM(safe.category) AS event_cat, 
    RTRIM(safe.incident_severity) AS event_sev, 
    safe.speed_kmh,
    safe.end_speed_kmh,
    safe.maxwaarde, 
    safe.road_name AS streetname, 
    safe.latitude AS lat, 
    safe.longitude AS lon,
    prec.RI_PWS_10 AS rain_intensity,
    temp.T_DRYB_10 AS temperature,
    wind.FF_SENSOR_10 AS windspeed
FROM 
    data_lake.precipitation prec
LEFT JOIN 
    data_lake.safe_driving safe ON prec.dtg = TO_TIMESTAMP(ROUND(EXTRACT(EPOCH FROM safe.event_start) / 600) * 600)
JOIN 
    data_lake.temperature temp ON prec.dtg = temp.dtg
JOIN 
    data_lake.wind wind ON prec.dtg = wind.dtg
WHERE 
    prec.RI_PWS_10 IS NOT NULL
    AND temp.T_DRYB_10 IS NOT NULL
    AND wind.FF_SENSOR_10 IS NOT NULL
    AND prec.dtg > '2018-01-01'
    AND (safe.is_valid IS NULL OR safe.is_valid = TRUE);
"""

In [5]:
cursor.execute(create_table_1)

In [6]:
conn.commit()
logging.info('Table 2 created and pushed to data warehouse.')

INFO: Table 2 created and pushed to data warehouse.


In [7]:
cursor.execute('SELECT * FROM group14_warehouse.timestream_table')
table_1 = pd.DataFrame(cursor.fetchall(), columns=['datetime_s', 'datetime_e', 'event_cat', 'event_sev', 'speed', 'end_speed',
                                                   'maxwaarde', 'streetname', 'lat', 'lon',
                                                   'rain_intensity', 'temperature', 'windspeed'])
table_1['datetime_s'] = pd.to_datetime(table_1['datetime_s'])
table_1['datetime_e'] = pd.to_datetime(table_1['datetime_e'])

print(table_1.head(10))

               datetime_s              datetime_e        event_cat event_sev  \
0                     NaT                     NaT             None      None   
1 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500            SPEED       SP1   
2 2018-01-01 00:24:42.500 2018-01-01 00:24:49.500            SPEED       SP1   
3 2018-01-01 00:22:54.500 2018-01-01 00:23:01.500            SPEED       SP1   
4 2018-01-01 00:24:33.800 2018-01-01 00:24:35.500  HARSH CORNERING       HC1   
5 2018-01-01 00:21:41.100 2018-01-01 00:21:43.500  HARSH CORNERING       HC1   
6 2018-01-01 00:25:07.900 2018-01-01 00:25:09.700  HARSH CORNERING       HC1   
7 2018-01-01 00:27:43.500 2018-01-01 00:27:49.500            SPEED       SP1   
8                     NaT                     NaT             None      None   
9 2018-01-01 00:51:05.600 2018-01-01 00:51:06.900  HARSH CORNERING       HC1   

       speed  end_speed  maxwaarde             streetname       lat       lon  \
0        NaN        NaN        NaN    

In [8]:
conn.close()
logging.info('Connection to database closed.')

INFO: Connection to database closed.
